# Do dado sintético ao modelo em produção — Bella Tavola 🍝
## MLOps Pipeline — Semana 2

Este caderno documenta o ciclo completo de gerenciamento do ciclo de vida de Machine Learning para a plataforma **Bella Tavola**:
1. Geração de dados sintéticos alinhados à regra de negócio do restaurante.
2. Treinamento de um `RandomForestClassifier` para mitigação de risco operacional em pedidos.
3. Publicação e versionamento do artefato binário no **Hugging Face Hub** (Model Registry).
4. Validação da inferência com cache local e garantia do contrato de dados.

--- 
# BLOCO 1 — Dados Sintéticos com Sabor de Restaurante 🍕

In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple

def gerar_dataset_bella_tavola(
    n_samples: int = 2000,
    seed: int = 42,
    proporcao_positivos: float = 0.25
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    Gera dataset sintético de pedidos de alto risco para o Bella Tavola.
    Target 1 indica um pedido potencialmente problemático/fraude.
    """
    if not (0.05 <= proporcao_positivos <= 0.95):
        raise ValueError("proporcao_positivos deve estar entre 0.05 e 0.95")

    rng = np.random.default_rng(seed)
    risco = rng.choice(
        [0, 1],
        size=n_samples,
        p=[1 - proporcao_positivos, proporcao_positivos]
    )

    # Simulação das regras de negócio do domínio
    valor_pedido = np.where(risco, rng.uniform(250, 900, n_samples), rng.uniform(25, 180, n_samples)).round(2)
    hora_pedido = np.where(risco, rng.choice([0, 1, 2, 3, 23], n_samples), rng.integers(11, 22, n_samples))
    num_itens = np.where(risco, rng.integers(6, 14, n_samples), rng.integers(1, 4, n_samples))
    historico_cancelamentos = np.where(risco, rng.integers(2, 5, n_samples), rng.integers(0, 1, n_samples))
    distancia_entrega = np.where(risco, rng.uniform(12, 45, n_samples), rng.uniform(0.5, 7, n_samples)).round(1)

    df = pd.DataFrame({
        "valor_pedido": valor_pedido,
        "hora_pedido": hora_pedido,
        "num_itens": num_itens,
        "historico_cancelamentos": historico_cancelamentos,
        "distancia_entrega": distancia_entrega,
        "target": risco
    })

    X = df.drop(columns=["target"]).values
    y = df["target"].values
    return df, X, y

df, X, y = gerar_dataset_bella_tavola(n_samples=2000, seed=42)
print("✅ Dataset gerado!")
print(df.groupby("target").mean().round(2))

✅ Dataset gerado!
        valor_pedido  hora_pedido  num_itens  historico_cancelamentos  \
target                                                                  
0             101.04        15.97       1.97                     0.00   
1             568.35         5.25       9.40                     2.98   

        distancia_entrega  
target                     
0                    3.85  
1                   28.90  


--- 
# BLOCO 2 — Treinamento e Serialização do Modelo 📊

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib as jl

# Split de treino/teste estratificado para manter a proporção das classes
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Métricas de Desempenho do Modelo:")
print(classification_report(y_test, y_pred, target_names=["legitimo", "risco"]))

# Serialização estável do artefato binário
jl.dump(model, "model.pkl")
print("✅ Artefato binário salvo como 'model.pkl'")

Métricas de Desempenho do Modelo:
              precision    recall  f1-score   support

    legitimo       1.00      1.00      1.00       295
       risco       1.00      1.00      1.00       105

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

✅ Artefato binário salvo como 'model.pkl'


--- 
# BLOCO 3 — Hugging Face Hub como Model Registry 🤗

In [ ]:
import os
import sklearn
from huggingface_hub import HfApi, login

print("🤗 [BLOCO 3] Autenticando no Hugging Face Hub...")

try:
    # Remove da memória do Jupyter qualquer variável antiga ou incorreta
    if "HF_TOKEN" in os.environ:
        del os.environ["HF_TOKEN"]
        
    # Se o token do cache estiver inválido, o login() abrirá um campo de texto seguro
    # dentro do próprio Notebook para você colar o seu token hf_... na hora!
    login() 
    
    api = HfApi()
    username = api.whoami()["name"]
    repo_id = f"{username}/mlops-bella-tavola-v1"
    
    print(f"📦 Vinculando ao Registry: {repo_id}")
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    
    # Geração do requirements do modelo
    with open("requirements.txt", "w") as f:
        f.write(f"scikit-learn=={sklearn.__version__}\njoblib==1.3.2\nnumpy=={np.__version__}\n")
        
    # Construindo o Model Card
    model_card = f"""---
language: pt
tags:
  - sklearn
  - classification
  - restaurante
  - mlops
---
# Bella Tavola — Classificador de Risco de Pedidos 🍝
"""
    with open("README_model.md", "w", encoding="utf-8") as f:
        f.write(model_card)
        
    print("📤 Enviando arquivos para o Hugging Face...")
    api.upload_file(path_or_fileobj="model.pkl", path_in_repo="model.pkl", repo_id=repo_id, repo_type="model")
    api.upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt", repo_id=repo_id, repo_type="model")
    api.upload_file(path_or_fileobj="README_model.md", path_in_repo="README.md", repo_id=repo_id, repo_type="model")
    
    if os.path.exists("README_model.md"): 
        os.remove("README_model.md")
        
    print(f"🚀 Sucesso! Repositório publicado em: https://huggingface.co/{repo_id}")

except Exception as e:
    print(f"❌ Erro na autenticação: {e}")

🤗 [BLOCO 3] Autenticando no Hugging Face Hub...


📦 Vinculando ao Registry: math04cezario/mlops-bella-tavola-v1
📤 Enviando arquivos para o Hugging Face...
🚀 Sucesso! Repositório publicado em: https://huggingface.co/math04cezario/mlops-bella-tavola-v1


--- 
# BLOCO 4 — Inferência Otimizada e Validação de Contrato 🛠️

In [ ]:
from huggingface_hub import hf_hub_download
import joblib
import os

def load_model(repo_id: str, filename: str = "model.pkl"):
    token = os.environ.get("HF_TOKEN")
    local_path = hf_hub_download(repo_id=repo_id, filename=filename, token=token)
    return joblib.load(local_path)

try:
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/mlops-bella-tavola-v1"
    modelo_remoto = load_model(repo_id)
    
    # Teste de fumaça focado na ordem das colunas da API
    amostra_teste = np.array([[750.0, 2, 12, 4, 42.0]])
    pred = modelo_remoto.predict(amostra_teste)[0]
    proba = modelo_remoto.predict_proba(amostra_teste)[0][1]
    
    print(f"✅ Teste de Contrato Concluído!")
    print(f"Predição: {pred} | Probabilidade de Risco: {proba:.4f}")
except Exception as e:
    print(f"❌ Erro na validação: {e}")

❌ Erro na validação: Invalid user token. If you didn't pass a user token, make sure you are properly logged in by executing `huggingface-cli login`, and if you did pass a user token, double-check it's correct.
